<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Mini_Projet_W9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCP + Agents AI Integration in Gemini
This notebook demonstrates an end-to-end agentic application orchestrating multiple MCP servers (filesystem, git, and custom tools) using Gemini.

In [ ]:
%%capture
# 1. Install dependencies
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "nest_asyncio" \
  "fastmcp>=2.0.0"

In [ ]:
# 2. Setup Environment
import nest_asyncio
nest_asyncio.apply()

# Confirm Node/NPM availability
!node --version
!npx --version

## Custom MCP Server
We create a local Python server using `FastMCP` to provide specialized tools for our agent.

In [ ]:
from pathlib import Path
import textwrap

server_path = Path("/content/custom_mcp_server.py")
server_path.write_text(textwrap.dedent("""
    from fastmcp import FastMCP
    from typing import Dict, List

    mcp = FastMCP(name="custom_ops")

    @mcp.tool
    def ping() -> str:
        \"\"\"Health check tool.\"\"\"
        return "pong"

    @mcp.tool
    def summarize_lines(lines: List[str]) -> Dict[str, int]:
        \"\"\"Returns counts about a list of lines.\"\"\"
        total = len(lines)
        nonempty = sum(1 for l in lines if l.strip())
        return {"total_lines": total, "nonempty_lines": nonempty}

    if __name__ == "__main__":
        mcp.run(transport="stdio")
"""), encoding="utf-8")

print("Wrote custom server to:", server_path)

## 3. Connect to MCP Servers
We will now initialize the `MultiServerMCPClient` to connect to our custom server and a filesystem server (using `npx`).

In [ ]:
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

# Define the directory for the filesystem server to manage
WORKDIR = "/content/workspace"
!mkdir -p {WORKDIR}

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "custom_ops": {
        "transport": "stdio",
        "command": "python",
        "args": [str(server_path)],
    },
}

client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)

# Get the tools in an async-friendly way for Colab
async def get_tools():
    async with client:
        return await client.get_tools()

tools = asyncio.run(get_tools())
print(f"Connected to {len(tools)} tools.")

## 4. Build the Gemini Agent
We use LangGraph to create a React-style agent that can invoke tools dynamically.

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

# Securely fetch the API Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# Initialize Gemini model
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

# Create the agent
agent_executor = create_react_agent(llm, tools)

print("Agent ready for action!")

## 5. Demonstration
Let's test the agent by asking it to create a file, check its health, and summarize the content.

In [ ]:
query = """
1. Use the custom_ops_ping tool to check health.
2. Create a file named 'hello.txt' in the workspace with 5 lines of text.
3. Use the custom_ops_summarize_lines tool to count the lines in that file.
"""

async def run_demo(query):
    async with client:
        async for chunk in agent_executor.astream({"messages": [("user", query)]}):
            print(chunk)
            print("-" * 20)

# Execute the demo
await run_demo(query)